# Estudo de caso: teste qui-quadrado de independência

Este notebook aplica o teste qui-quadrado de independência ao conjunto `survey.csv`, seguindo o roteiro de `GCC1625_06_qui_quadrado`. A pergunta é: **há evidência de associação entre sexo e nível de exercício físico entre os estudantes da amostra?**

O notebook geral `06_testes_hipoteses_qui_quadrado.ipynb` apresenta as três modalidades do teste e suas funções auxiliares. Aqui o foco é um estudo de caso completo: organizar os dados, verificar as condições, aplicar o teste e interpretar o padrão observado.


## 1. Hipóteses e preparação dos dados

As variáveis são categóricas: `Sex` registra o sexo informado e `Exer` registra o nível de exercício físico. Para o teste de independência:

$$H_0: \text{sexo e nível de exercício são independentes na população}$$

$$H_a: \text{sexo e nível de exercício são associados na população}$$

O arquivo usa abreviações para exercício: `Freq` (frequente), `Some` (algum exercício) e `None` (nenhum exercício).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

ALPHA = 0.05

arquivo = Path('../data/survey.csv')
dados = pd.read_csv(arquivo, index_col=0, na_values='NA')

dados = dados[['Sex', 'Exer']].dropna().copy()
dados['Sex'] = pd.Categorical(dados['Sex'], categories=['Female', 'Male'])
dados['Exer'] = pd.Categorical(dados['Exer'], categories=['Freq', 'Some', 'None'])

dados.head(), dados.shape


## 2. Tabela observada e descrição do padrão

Cada linha do conjunto representa um estudante. A tabela de contingência reúne as contagens observadas $$O_{ij}$$ para cada combinação de sexo e nível de exercício. Além das contagens, as proporções por linha ajudam a descrever a distribuição do exercício dentro de cada grupo.


In [ ]:
tabela = pd.crosstab(dados['Sex'], dados['Exer'])
tabela


In [ ]:
proporcoes_linha = tabela.div(tabela.sum(axis=1), axis=0)
proporcoes_linha.round(3)


In [ ]:
ax = proporcoes_linha.plot(
    kind='bar', stacked=True, figsize=(8, 4),
    color=['#4C78A8', '#72B7B2', '#F58518'],
)
ax.set(
    title='Distribuição do nível de exercício dentro de cada grupo',
    xlabel='Sexo',
    ylabel='Proporção',
)
ax.legend(title='Exercício', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()


**Leitura descritiva.** O gráfico e as proporções descrevem a amostra, mas não decidem a questão inferencial. Diferenças nas proporções podem surgir por flutuação amostral mesmo se sexo e exercício forem independentes na população. O teste qui-quadrado compara as contagens observadas com as que seriam esperadas sob essa independência.


## 3. Condições de aplicabilidade

A primeira condição é que as observações sejam independentes. Ela depende do desenho da pesquisa: por exemplo, cada estudante deve aparecer uma única vez, e uma resposta não deve influenciar a de outro estudante. Nenhum cálculo sobre a tabela pode confirmar essa condição.

A segunda condição é a adequação da aproximação qui-quadrado. Inspecionaremos as frequências esperadas; como regra prática, elas devem ser pelo menos 5.


In [ ]:
chi2_stat, p_valor, gl, esperadas = chi2_contingency(tabela, correction=False)
esperadas_df = pd.DataFrame(esperadas, index=tabela.index, columns=tabela.columns)

esperadas_df.round(2)


In [ ]:
minima_esperada = esperadas_df.to_numpy().min()
print(f'Menor frequência esperada: {minima_esperada:.2f}')
print(f'Todas as frequências esperadas são pelo menos 5? {minima_esperada >= 5}')


**Leitura das condições.** Se todas as frequências esperadas forem pelo menos 5, a regra prática para a aproximação qui-quadrado está satisfeita. Isso não substitui a avaliação da independência das observações. Caso as frequências esperadas fossem pequenas, seria necessário reconsiderar a análise, por exemplo combinando categorias justificadamente ou usando métodos exatos ou simulação, conforme a estrutura da tabela.


## 4. Teste qui-quadrado de independência

A função `chi2_contingency` retorna a estatística $$\chi^2$$, o p-valor, os graus de liberdade e as frequências esperadas. Usamos `correction=False` para aplicar o teste de Pearson sem a correção de continuidade de Yates, tal como nos cálculos da nota.


In [ ]:
print(f'Estatística qui-quadrado: {chi2_stat:.3f}')
print(f'Graus de liberdade: {gl}')
print(f'p-valor: {p_valor:.4f}')

if p_valor <= ALPHA:
    print('Decisão: rejeitamos H0; há evidência de associação entre as variáveis.')
else:
    print('Decisão: não rejeitamos H0; não há evidência suficiente de associação.')


**Leitura do resultado.** Compare o p-valor com $$\alpha = 0{,}05.$$ Se $$p \leq \alpha,$$ os dados fornecem evidência contra a hipótese de independência. Se $$p > \alpha,$$ não rejeitamos essa hipótese. Em nenhum dos casos o p-valor mede a intensidade da associação nem permite concluir causalidade.


## 5. Tamanho do efeito: V de Cramér

Para uma tabela de dimensão geral, o V de Cramér é

$$V = \sqrt{\frac{\chi^2}{N\min(r-1,c-1)}}.$$

Ele complementa o p-valor ao quantificar a magnitude do padrão de associação na tabela. Sua interpretação deve considerar o contexto da pesquisa, o formato da tabela e as proporções observadas.


In [ ]:
n = tabela.to_numpy().sum()
r, c = tabela.shape
v_cramer = np.sqrt(chi2_stat / (n * min(r - 1, c - 1)))

print(f'N = {n}')
print(f'V de Cramér = {v_cramer:.3f}')


## 6. Onde aparece o padrão?

O teste global resume todos os desvios em uma única estatística. Para descrever o padrão da amostra, comparamos as contagens observadas com as esperadas e calculamos os resíduos brutos:

$$d_{ij}=O_{ij}-E_{ij}.$$

Um resíduo positivo indica sobre-representação daquela combinação em relação ao cenário de independência; um resíduo negativo indica sub-representação. Essa leitura é descritiva: o sinal, por si só, não é um teste separado para cada célula.


In [ ]:
residuos_brutos = tabela - esperadas_df

print('Frequências observadas:')
display(tabela)

print('Frequências esperadas sob independência:')
display(esperadas_df.round(2))

print('Resíduos brutos (observada - esperada):')
display(residuos_brutos.round(2))


**Leitura dos resíduos.** Compare os sinais dos resíduos com as proporções por linha apresentadas no início. Células positivas mostram combinações mais frequentes do que esperaríamos se as variáveis fossem independentes; células negativas mostram combinações menos frequentes. Como os totais das linhas e colunas são mantidos, os resíduos de uma mesma linha ou coluna se compensam. Portanto, eles não devem ser tratados como evidências independentes.


## 7. Conclusão reportável

Uma conclusão deve indicar o teste, a estatística, os graus de liberdade, o p-valor, o tamanho do efeito e a interpretação no contexto. A célula abaixo produz uma frase-base; complete-a com a descrição do padrão observada nas proporções e nos resíduos.


In [ ]:
decisao = (
    'rejeitamos a hipótese de independência'
    if p_valor <= ALPHA
    else 'não rejeitamos a hipótese de independência'
)

print(
    f'Foi aplicado um teste qui-quadrado de independência entre sexo e nível '
    f'de exercício: χ²({gl}) = {chi2_stat:.2f}, p = {p_valor:.4f}, '
    f'V de Cramér = {v_cramer:.3f}. Ao nível de 5%, {decisao}.'
)


## Exercício de extensão

Repita a análise para as variáveis `Smoke` e `Exer` do arquivo original. Antes de aplicar o teste, inspecione cuidadosamente as frequências esperadas: algumas categorias de tabagismo são pouco frequentes. Discuta se a aproximação qui-quadrado parece adequada e o que poderia ser feito se não for.
